In [3]:
import pybamm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import dfols
import signal
from tqdm import tqdm
from scipy.integrate import solve_ivp
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy import interpolate, integrate
from stopit import threading_timeoutable as timeoutable
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from batfuns import *
plt.rcParams = set_rc_params(plt.rcParams)
import winsound
from pybamm import exp, constants, Parameter
import pickle
import matplotlib as mpl
pd.options.mode.chained_assignment = None


eSOH_DIR = "../data/esoh_R/"
oCV_DIR = "../data/ocv/"
cyc_DIR = "../data/cycling/"
fig_DIR = "../figures/figures_paper/"
res_DIR = "../data/results_paper/"
resistance_DIR = "../data/resistance/"
%matplotlib widget

In [4]:
parameter_values = get_parameter_values()

spm = pybamm.lithium_ion.SPM(
    {
        "SEI": "ec reaction limited",
        "loss of active material": "stress-driven",
        "lithium plating": "irreversible",
        "stress-induced diffusion": "false",
    }
)
# spm.print_parameter_info()
param=spm.param

In [19]:
def get_Rs(sols):
    Rs = []
    for i in range(len(sols)):
        sol_long = sols[i]
        I = sol_long["Current [A]"].entries
        V = sol_long["Terminal voltage [V]"].entries
        idxi1 = np.where((np.diff(I)>5) & (I[:-1]>-2))[0]
        idx = idxi1[0]
        R = -(V[idx+1] - V[idx])/(I[idx+1] - I[idx])
        Rs.append(R)
    Rs = np.array(Rs)
    return Rs

In [20]:
def get_acap(sols):
    Qappd = []
    Qappc = []
    for i in range(len(sols)):
        sol_long = sols[i]
        Q = -sol_long['Discharge capacity [A.h]'].entries
        Q = Q - Q[0]
        Q2 = max(Q)
        Q1 = max(Q) - Q[-1]
        Qappd.append(Q1)
        Qappc.append(Q2)
    Qappd = np.array(Qappd)
    Qappc = np.array(Qappc)
    return Qappd

In [7]:
def get_exp(sols):
    b1=296.32
    b2=9608263.5
    b3=476.53
    b4=0.0
    del_sei = []
    del_li = []
    es_n = []
    es_p = []
    for i in range(len(sols)):
        df = sols[i]
        del_sei.append(df["X-averaged SEI thickness [m]"].entries[0])
        # Plated Lithium thickness
        del_li.append(df["X-averaged lithium plating thickness [m]"].entries[0])
        # negative electode inactive material
        es_n.append(df["X-averaged negative electrode active material volume fraction"].entries[0])
        # positive electrode inactive material 
        es_p.append(df["X-averaged positive electrode active material volume fraction"].entries[0])

    del_sei = np.array(del_sei)
    del_sei = del_sei - del_sei[0]
    del_li = np.array(del_li)
    es_n = np.array(es_n)
    es_ic_n = -es_n + es_n[0]
    es_p = np.array(es_n) 
    es_ic_p = -es_p + es_p[0]
    irrev_exp = (b1*del_sei*1e6+b2*del_li**2*1e12+b3*es_ic_n+b4*es_ic_p)
    return irrev_exp

In [18]:
def get_cn(sols):
    Cn = []
    for i in range(len(sols)):
        sol_long = sols[i]
        Cn1 = sol_long.all_summary_variables
        Cn2 = Cn1[0]['C_n']
        Cn.append(Cn2)
    Cn = np.array(Cn)
    return Cn

In [9]:
with open('hot_sim_2C_5psi_sim_0.pickle', 'rb') as handle:
    sols1 = pickle.load(handle)
with open('hot_sim_2C_5psi_sim_b_1.pickle', 'rb') as handle:
    sols2 = pickle.load(handle)
with open('hot_sim_2C_5psi_sim_b_2.pickle', 'rb') as handle:
    sols3 = pickle.load(handle)
with open('hot_sim_2C_5psi_sim_b_3.pickle', 'rb') as handle:
    sols4 = pickle.load(handle)
with open('hot_sim_2C_5psi_sim_b_4.pickle', 'rb') as handle:
    sols5 = pickle.load(handle)
with open('hot_sim_2C_5psi_sim_b_5.pickle', 'rb') as handle:
    sols6 = pickle.load(handle)

In [1]:
xs = [0.5,0.6,0.7,0.8,0.9,1.0,1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9,2]

In [2]:
len(xs)

16

In [21]:
# Rs0 = get_Rs(sols0)
Rs1 = get_Rs(sols1)
Rs2 = get_Rs(sols2)
Rs3 = get_Rs(sols3)
Rs4 = get_Rs(sols4)
Rs5 = get_Rs(sols5)
Rs6 = get_Rs(sols6)

In [22]:
# Q0 = get_acap(sols0)
Q1 = get_acap(sols1)
Q2 = get_acap(sols2)
Q3 = get_acap(sols3)
Q4 = get_acap(sols4)
Q5 = get_acap(sols5)
Q6 = get_acap(sols6)

In [23]:
E1 = get_exp(sols1)
E2 = get_exp(sols2)
E3 = get_exp(sols3)
E4 = get_exp(sols4)
E5 = get_exp(sols5)
E6 = get_exp(sols6)


In [24]:
Cn1 = get_cn(sols1)
Cn2 = get_cn(sols2)
Cn3 = get_cn(sols3)
Cn4 = get_cn(sols4)
Cn5 = get_cn(sols5)
Cn6 = get_cn(sols6)


In [ ]:
# Q1,Q2,Q3,Q4,Q5,Q6

In [39]:
with open("hot_sims.pickle", 'wb') as file:
    pickle.dump(
        [Q1,Q2,Q3,Q4,Q5,Q6,
        Rs1,Rs2,Rs3,Rs4,Rs5,Rs6,
        E1,E2,E3,E4,E5,E6,
        Cn1,Cn2,Cn3,Cn4,Cn5,Cn6],
        file
    )